# LFR-Net (Grosz & Jain, TIFS 2023) - Latent Enhancement + Segmentation
Paper-aligned: Squeeze U-Net, **2 output channels** (enhanced image + ridge image), loss = MSE(enhanced, clean) + MSE(ridge, ridge_GT) with equal weights, Adam lr 1e-3, polynomial LR decay, 11 epochs, segmentation mask from the ridge channel.

Paper vs this notebook (jo alag hai):
- Paper synthetic augmentation se noisy banata hai; yahan tumhare ready noisy/clean pairs use ho rahe hain.
- Ridge ground truth paper me VeriFinger v12.3 se aata hai (commercial). Yahan clean image se open-source adaptive threshold se banaya hai.
- Paper exact layer sizes nahi deta, isliye Squeeze U-Net ka standard Fire-module design use kiya hai.

In [ ]:
# 1. Download + extract dataset (re-extracts if the previous extraction was incomplete)
!pip -q install gdown
import os, zipfile, shutil, gdown
FILE_ID  = "1njl-hWUG7OEQVilJ4f8w77YuZe7EXrtM"
ZIP_PATH = "/kaggle/working/dataset.zip"
EXTRACT  = "/kaggle/working/dataset"
EXPECTED = 4984                      # total files inside the zip (2490 clean + 2490 noisy + metadata)

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(f"{EXTRACT}/noise_project_v2") != EXPECTED:
    if not (os.path.exists(ZIP_PATH) and zipfile.is_zipfile(ZIP_PATH)):
        gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)
    shutil.rmtree(EXTRACT, ignore_errors=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT)
n = n_files(f"{EXTRACT}/noise_project_v2")
print("files extracted:", n, "(expected", EXPECTED, ")")
assert n == EXPECTED, "extraction incomplete - check disk space with !df -h /kaggle/working"

In [ ]:
# 2. Config, imports, pairing and official split
import os, re, glob, random, math, time
import numpy as np, pandas as pd, cv2
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_fn
from tqdm.auto import tqdm

ROOT      = "/kaggle/working/dataset/noise_project_v2"
CLEAN_DIR = f"{ROOT}/clean_500"
NOISY_DIR = f"{ROOT}/noisy_500_v2"
OUT_DIR   = "/kaggle/working/lfrnet_paper"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- paper settings ----
EPOCHS, LR, POWER = 11, 1e-3, 0.9      # paper: 11 epochs, Adam lr 1e-3, polynomial schedule
# ---- training settings (paper does not specify) ----
PATCH, BATCH, MULT = 384, 8, 4         # MULT = patches per image per epoch
# ---- ridge ground-truth (VeriFinger replacement) ----
RIDGE_BLOCK, RIDGE_C = 17, 4           # adaptive threshold block size / constant
# ---- segmentation (paper Sec. III-A) ----
SEG_BLUR, SEG_THR, SEG_K, SEG_REP, SEG_MIN_AREA = 5, 150, 9, 3, 10000

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ON = DEVICE == "cuda"
print("device:", DEVICE)

clean_files = sorted(glob.glob(f"{CLEAN_DIR}/**/*.png", recursive=True))
noisy_files = sorted(glob.glob(f"{NOISY_DIR}/**/*.png", recursive=True))
print("clean:", len(clean_files), "| noisy:", len(noisy_files))

PAT = re.compile(r"(\d+_[A-Za-z]_500_[A-Za-z]+_\d+)")
clean_map = {os.path.splitext(os.path.basename(p))[0]: p for p in clean_files}
pairs = []
for n in noisy_files:
    m = PAT.search(os.path.basename(n))
    if m and m.group(1) in clean_map:
        pairs.append((n, clean_map[m.group(1)], m.group(1).split("_")[0]))   # (noisy, clean, participant)
print("total pairs:", len(pairs))

sp = pd.read_csv(f"{ROOT}/b6000_split.csv")
key_lab = dict(zip(sp["key"].astype(str), sp["split"].astype(str).str.lower().str.strip()))
buckets, missing = {"train": [], "val": [], "test": []}, 0
for p in pairs:
    lab = key_lab.get(os.path.splitext(os.path.basename(p[1]))[0])
    if lab in buckets: buckets[lab].append(p)
    else: missing += 1
assert missing == 0, f"{missing} pairs not found in split csv"
train_pairs, val_pairs, test_pairs = buckets["train"], buckets["val"], buckets["test"]
print(f"train/val/test: {len(train_pairs)}/{len(val_pairs)}/{len(test_pairs)}  (expected 1764/388/338)")

subj = lambda L: {p[2] for p in L}
print("participant overlap:", len(subj(train_pairs) & subj(test_pairs)),
      len(subj(train_pairs) & subj(val_pairs)), len(subj(val_pairs) & subj(test_pairs)))

In [ ]:
# 3. Ridge ground truth + dataset + loaders
def load_gray(path):
    return np.asarray(Image.open(path).convert("L"), dtype=np.float32) / 255.0

def make_ridge(y):
    """clean image [0,1] -> ridge image [0,1] (dark ridges = 0, background = 1). Replaces VeriFinger ridge map."""
    u8 = (np.clip(y, 0, 1) * 255).astype(np.uint8)
    u8 = cv2.GaussianBlur(u8, (0, 0), 1.0)
    r = cv2.adaptiveThreshold(u8, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, RIDGE_BLOCK, RIDGE_C)
    return r.astype(np.float32) / 255.0

class PairPatchDataset(Dataset):
    def __init__(self, pairs, patch=PATCH, length_mult=1):
        self.pairs, self.patch, self.mult = pairs, patch, length_mult
    def __len__(self): return len(self.pairs) * self.mult
    def __getitem__(self, i):
        noisy_p, clean_p, _ = self.pairs[i % len(self.pairs)]
        x, y = load_gray(noisy_p), load_gray(clean_p)
        assert x.shape == y.shape
        P = self.patch; H, W = x.shape
        mask = np.ones_like(x)
        ph, pw = max(0, P - H), max(0, P - W)
        if ph or pw:                                   # small image: white pad, mask = 0 (ignored in loss)
            x = np.pad(x, ((0, ph), (0, pw)), constant_values=1.0)
            y = np.pad(y, ((0, ph), (0, pw)), constant_values=1.0)
            mask = np.pad(mask, ((0, ph), (0, pw)), constant_values=0.0)
            H, W = x.shape
        for _ in range(10):                            # find a crop that contains fingerprint
            t, l = random.randint(0, H - P), random.randint(0, W - P)
            if (1 - y[t:t+P, l:l+P]).mean() > 0.10:
                break
        x, y, mask = x[t:t+P, l:l+P], y[t:t+P, l:l+P], mask[t:t+P, l:l+P]
        if random.random() < 0.5: x, y, mask = x[:, ::-1], y[:, ::-1], mask[:, ::-1]
        if random.random() < 0.5: x, y, mask = x[::-1], y[::-1], mask[::-1]
        x, y, mask = np.ascontiguousarray(x), np.ascontiguousarray(y), np.ascontiguousarray(mask)
        r = make_ridge(y)
        f = lambda a: torch.from_numpy(a).unsqueeze(0)
        return f(x), f(y), f(r), f(mask)

NW = min(4, os.cpu_count() or 2)
train_dl = DataLoader(PairPatchDataset(train_pairs, length_mult=MULT), batch_size=BATCH, shuffle=True,
                      num_workers=NW, drop_last=True, pin_memory=True, persistent_workers=NW > 0)
print("iterations per epoch:", len(train_dl))

# sanity: check that the ridge ground truth looks right
x, y, r, m = next(iter(train_dl))
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, t in zip(ax, [x[0, 0], y[0, 0], r[0, 0]], ["noisy", "clean", "ridge GT"]):
    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(t); a.axis("off")
plt.show()

In [ ]:
# 4. Squeeze U-Net (Fire modules), 2 output channels: [enhanced, ridge]
class Fire(nn.Module):
    def __init__(self, cin, squeeze, expand):
        super().__init__()
        self.sq  = nn.Sequential(nn.Conv2d(cin, squeeze, 1), nn.ReLU(inplace=True))
        self.e1  = nn.Conv2d(squeeze, expand, 1)
        self.e3  = nn.Conv2d(squeeze, expand, 3, padding=1)
        self.bn  = nn.BatchNorm2d(2 * expand)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        z = self.sq(x)
        return self.act(self.bn(torch.cat([self.e1(z), self.e3(z)], 1)))   # out channels = 2*expand

class SqueezeUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.e1 = Fire(32, 16, 32)                                         # 64  ch, full res
        self.e2 = nn.Sequential(Fire(64, 32, 64),  Fire(128, 32, 64))      # 128 ch, 1/2
        self.e3 = nn.Sequential(Fire(128, 48, 128), Fire(256, 48, 128))    # 256 ch, 1/4
        self.e4 = nn.Sequential(Fire(256, 64, 192), Fire(384, 64, 192))    # 384 ch, 1/8
        self.b  = nn.Sequential(Fire(384, 80, 256), Fire(512, 80, 256))    # 512 ch, 1/16
        self.pool = nn.MaxPool2d(2)
        self.u4 = nn.ConvTranspose2d(512, 384, 2, stride=2); self.d4 = Fire(768, 64, 192)
        self.u3 = nn.ConvTranspose2d(384, 256, 2, stride=2); self.d3 = Fire(512, 48, 128)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, stride=2); self.d2 = Fire(256, 32, 64)
        self.u1 = nn.ConvTranspose2d(128, 64, 2, stride=2);  self.d1 = Fire(128, 16, 32)
        self.out = nn.Conv2d(64, 2, 1)
    def forward(self, x):
        s1 = self.e1(self.stem(x))
        s2 = self.e2(self.pool(s1))
        s3 = self.e3(self.pool(s2))
        s4 = self.e4(self.pool(s3))
        b  = self.b(self.pool(s4))
        d = self.d4(torch.cat([self.u4(b), s4], 1))
        d = self.d3(torch.cat([self.u3(d), s3], 1))
        d = self.d2(torch.cat([self.u2(d), s2], 1))
        d = self.d1(torch.cat([self.u1(d), s1], 1))
        return torch.sigmoid(self.out(d))      # (B,2,H,W) in [0,1]  (paper: [0,255])

model = SqueezeUNet().to(DEVICE)
print("params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))

In [ ]:
# 5. Loss, inference, segmentation, metrics
def masked_mse(a, b, m):
    return (((a - b) ** 2) * m).sum() / m.sum().clamp(min=1)

def lfr_loss(pred, y, r, m):
    """paper: MSE(enhanced, clean) + MSE(ridge, ridge_GT), equal weight"""
    return masked_mse(pred[:, 0:1], y, m) + masked_mse(pred[:, 1:2], r, m)

def pad_to(x, mult=16):
    H, W = x.shape[-2:]
    return F.pad(x, (0, (-W) % mult, 0, (-H) % mult), value=1.0), H, W

@torch.no_grad()
def enhance_full(model, noisy_np):
    """full image -> (enhanced, ridge), both float [0,1]"""
    model.eval()
    x = torch.from_numpy(noisy_np)[None, None].to(DEVICE)
    x, H, W = pad_to(x)
    out = model(x)[0, :, :H, :W].clamp(0, 1).cpu().numpy()
    return out[0], out[1]

def ridge_to_mask(ridge):
    """paper Sec. III-A: Gaussian(5,5) -> threshold 150 -> closing(9,9) x3 -> opening(9,9) x3;
    if mask area < 10,000 px the whole image is used."""
    r255 = (np.clip(ridge, 0, 1) * 255).astype(np.uint8)
    r255 = cv2.GaussianBlur(r255, (SEG_BLUR, SEG_BLUR), 0)
    b = (r255 < SEG_THR).astype(np.uint8)              # ridges are dark in our ridge image
    k = np.ones((SEG_K, SEG_K), np.uint8)
    for _ in range(SEG_REP): b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, k)
    for _ in range(SEG_REP): b = cv2.morphologyEx(b, cv2.MORPH_OPEN, k)
    if b.sum() < SEG_MIN_AREA: b = np.ones_like(b)
    return b.astype(np.float32)

def segment(enh, ridge):
    m = ridge_to_mask(ridge)
    return enh * m + (1 - m), m                         # white background outside the mask

def metrics(a, b):                                      # RMSE / PSNR / SSIM on 0-255 scale
    a255, b255 = a * 255.0, b * 255.0
    mse = ((a255 - b255) ** 2).mean()
    psnr = 10 * math.log10(255.0 ** 2 / mse) if mse > 0 else 99.0
    return math.sqrt(mse), psnr, ssim_fn(a255, b255, data_range=255.0)

val_sub = random.Random(0).sample(val_pairs, min(100, len(val_pairs)))
def val_eval(model, pairs):
    out = []
    for a, b, _ in pairs:
        e, _r = enhance_full(model, load_gray(a))
        out.append(metrics(e, load_gray(b)))
    return tuple(np.mean(out, axis=0))                  # (RMSE, PSNR, SSIM)

bl = np.mean([metrics(load_gray(a), load_gray(b)) for a, b, _ in val_sub], axis=0)
print(f"baseline (noisy vs clean) | RMSE {bl[0]:.2f} | PSNR {bl[1]:.2f} | SSIM {bl[2]:.4f}")

In [ ]:
# 6. Train (Adam lr 1e-3, polynomial decay)
CKPT = f"{OUT_DIR}/last_ckpt.pt"
model = SqueezeUNet().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
total_steps = EPOCHS * len(train_dl)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (1 - min(s, total_steps) / total_steps) ** POWER)
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ON)
start_ep, best_ssim, skipped, hist = 0, -1, 0, []

if os.path.exists(CKPT):                                # auto-resume
    ck = torch.load(CKPT, map_location=DEVICE)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    start_ep, best_ssim, hist = ck["ep"], ck["best"], ck["hist"]
    print("resumed from epoch", start_ep)

t_start = time.time()
for ep in range(start_ep, EPOCHS):
    model.train(); run, cnt = 0.0, 0
    for x, y, r, m in tqdm(train_dl, desc=f"epoch {ep+1}/{EPOCHS}", leave=False):
        x, y, r, m = x.to(DEVICE), y.to(DEVICE), r.to(DEVICE), m.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ON):
            pred = model(x)
        loss = lfr_loss(pred.float(), y, r, m)          # loss in fp32
        if not torch.isfinite(loss):
            skipped += 1; opt.zero_grad(set_to_none=True); continue
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item(); cnt += 1
    vr, vp, vs = val_eval(model, val_sub)
    hist.append(dict(epoch=ep+1, train_loss=run/max(cnt, 1), val_rmse=vr, val_psnr=vp, val_ssim=vs))
    print(f"epoch {ep+1}/{EPOCHS} | loss {run/max(cnt,1):.4f} | val RMSE {vr:.2f} PSNR {vp:.2f} SSIM {vs:.4f} "
          f"| skipped {skipped} | {(time.time()-t_start)/60:.0f} min")
    if vs > best_ssim:
        best_ssim = vs; torch.save(model.state_dict(), f"{OUT_DIR}/lfrnet_best.pt")
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                    scaler=scaler.state_dict(), ep=ep+1, best=best_ssim, hist=hist), CKPT)
    pd.DataFrame(hist).to_csv(f"{OUT_DIR}/train_history.csv", index=False)
print("best val SSIM:", best_ssim)

In [ ]:
# 7. Test: enhanced + segmented outputs, metrics vs clean, save images (500 ppi)
model = SqueezeUNet().to(DEVICE)
model.load_state_dict(torch.load(f"{OUT_DIR}/lfrnet_best.pt", map_location=DEVICE))

DIRS = {k: f"{OUT_DIR}/test_out/{k}" for k in ["noisy", "enhanced", "segmented", "ridge", "mask", "clean"]}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)
save = lambda a, d, n: Image.fromarray((np.clip(a, 0, 1) * 255).round().astype(np.uint8)).save(f"{DIRS[d]}/{n}.png", dpi=(500, 500))

rows = []
for noisy_p, clean_p, sid in tqdm(test_pairs):
    name = os.path.splitext(os.path.basename(clean_p))[0]
    x, c = load_gray(noisy_p), load_gray(clean_p)
    e, r = enhance_full(model, x)
    s, m = segment(e, r)
    r0, p0, s0 = metrics(x, c); r1, p1, s1 = metrics(e, c); r2, p2, s2 = metrics(s, c)
    rows.append(dict(name=name, participant=sid, device=name.split("_")[1],
                     rmse_noisy=r0, psnr_noisy=p0, ssim_noisy=s0,
                     rmse_enh=r1, psnr_enh=p1, ssim_enh=s1,
                     rmse_seg=r2, psnr_seg=p2, ssim_seg=s2, mask_frac=m.mean()))
    for a, d in [(x, "noisy"), (e, "enhanced"), (s, "segmented"), (r, "ridge"), (m, "mask"), (c, "clean")]:
        save(a, d, name)

res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/test_metrics.csv", index=False)
cols = [f"{k}_{v}" for v in ["noisy", "enh", "seg"] for k in ["rmse", "psnr", "ssim"]]
print("ALL TEST SET\n", res[cols].mean().round(4))
print("\nDEVICE-WISE\n", res.groupby("device")[cols].mean().round(3))

In [ ]:
# 8. Curves + visuals (noisy | enhanced | ridge | mask | segmented | clean)
h = pd.read_csv(f"{OUT_DIR}/train_history.csv")
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(h.epoch, h.train_loss); ax[0].set_title("train loss")
ax[1].plot(h.epoch, h.val_ssim);   ax[1].set_title("val SSIM")
ax[2].plot(h.epoch, h.val_psnr);   ax[2].set_title("val PSNR")
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/curves.png", dpi=150); plt.show()

def show(df, n=4, fname="visuals.png"):
    sel = df.head(n) if len(df) <= n else df.sample(n, random_state=1) if fname == "visuals.png" else df.head(n)
    keys = ["noisy", "enhanced", "ridge", "mask", "segmented", "clean"]
    fig, ax = plt.subplots(len(sel), len(keys), figsize=(3 * len(keys), 4 * len(sel)))
    ax = np.atleast_2d(ax)
    for i, (_, rr) in enumerate(sel.iterrows()):
        for j, k in enumerate(keys):
            ax[i, j].imshow(np.asarray(Image.open(f"{DIRS[k]}/{rr['name']}.png")), cmap="gray", vmin=0, vmax=255)
            ax[i, j].set_title(k if i else f"{k}", fontsize=9); ax[i, j].axis("off")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/{fname}", dpi=150); plt.show()

show(res, 4, "visuals.png")
show(res.sort_values("ssim_enh"), 4, "visuals_worst.png")

In [ ]:
# 9. (Optional) NFIQ2 on noisy / enhanced / segmented / clean test images
import subprocess, shutil
if shutil.which("nfiq2") is None:
    osr = open("/etc/os-release").read()
    ver = re.search(r'VERSION_ID="?([\d.]+)', osr).group(1)
    tag = {"20.04": "ubuntu20", "22.04": "ubuntu22", "24.04": "ubuntu24"}[ver]
    url = f"https://github.com/usnistgov/NFIQ2/releases/download/v2.3.0/nfiq2_2.3.0-1_{tag}_amd64.deb"
    fn = f"/kaggle/working/{url.split('/')[-1]}"
    subprocess.run(["wget", "-q", "-O", fn, url], check=True)
    print(subprocess.run(["dpkg", "-i", fn], capture_output=True, text=True).stderr[-500:])

def run_nfiq2(group):
    out_csv = f"{OUT_DIR}/nfiq2_{group}.csv"
    r = subprocess.run(["nfiq2", "-F", "-i", DIRS[group], "-o", out_csv], capture_output=True, text=True)
    assert os.path.exists(out_csv), r.stderr[-1000:]
    d = pd.read_csv(out_csv)
    fcol = next(c for c in d.columns if "file" in c.lower())
    qcol = next(c for c in d.columns if "quality" in c.lower() or "score" in c.lower())
    d["name"] = d[fcol].map(lambda p: os.path.splitext(os.path.basename(str(p)))[0])
    d[group] = pd.to_numeric(d[qcol], errors="coerce")
    d.loc[(d[group] < 0) | (d[group] > 100), group] = np.nan
    return d.set_index("name")[group]

nf = pd.DataFrame({g: run_nfiq2(g) for g in ["noisy", "enhanced", "segmented", "clean"]}).dropna()
nf.to_csv(f"{OUT_DIR}/nfiq2_all.csv")
print(f"n = {len(nf)}")
print(nf.agg(["mean", "median", "std"]).round(2))
print("% >= 60:\n", (nf >= 60).mean().mul(100).round(1))